In [2]:
library(plyr)
library(dplyr)
library(tidyr)
library(lme4)
library(lmerTest)
library(interactions)
library(ggplot2)
library(emmeans)
library(rempsyc)

------------------------------------------------------------------------------

You have loaded plyr after dplyr - this is likely to cause problems.
If you need functions from both plyr and dplyr, please load plyr first, then dplyr:
library(plyr); library(dplyr)

------------------------------------------------------------------------------


Attaching package: 'plyr'


The following objects are masked from 'package:dplyr':

    arrange, count, desc, failwith, id, mutate, rename, summarise,
    summarize


Loading required package: Matrix


Attaching package: 'Matrix'


The following objects are masked from 'package:tidyr':

    expand, pack, unpack



Attaching package: 'lmerTest'


The following object is masked from 'package:lme4':

    lmer


The following object is masked from 'package:stats':

    step


Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'

Suggested APA citation: Th<U+00E9>riault, R. (2023). rempsyc: Co

# TF measures predicted by acc * soc * age

## Load data

In [5]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'power_early',
    'ITPS_early',
    'ICPS_early_DLPFC_collapsed',
    'ICPS_early_MOTOR_collapsed',
    'ICPS_early_OCC_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))

tf_data$soc <- as.factor(tf_data$soc)
tf_data$sub <- as.factor(tf_data$sub)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_20_05_2026_14_17_15.csv"


In [19]:
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
filter(tf_data, sub == 3000314)$age_m[1] # now 154

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_20_05_2026_14_17_15.csv"


[1] 154

In [20]:
filter(tf_data, sub == 3000252)$age_m[1] # now 168

[1] 168

In [21]:
filter(tf_data, sub == 3000269)$age_m[1] # now 161

[1] 161

In [22]:
filter(tf_data, sub == 3000014)$age_m[1] # now 160

[1] NA

In [16]:
filter(tf_data, sub == 3000014)

Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"
Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"
Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"
Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
"number of rows of result is not a multiple of vector length (arg 2)"


sub,age_m,sex,first_soc,dp_inperson,soc,acc,X6_or_more_err,ERN_min_CRN,ERN_min_CRN_laplacian,...,dyadb_i4_s1_r1_e1,dyadb_i5_s1_r1_e1,dyadb_i6_s1_r1_e1,dyadb_i7_s1_r1_e1,dyadb_i8_s1_r1_e1,dyadb_i9_s1_r1_e1,selfnowa_i1_s1_r1_e2,selfnowa_i2_s1_r1_e2,selfnowa_i3_s1_r1_e2,selfnowa_i4_s1_r1_e2
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>


In [24]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
  "power_early", 
  "ITPS_early", 
  "ICPS_early_DLPFC_collapsed", 
  "ICPS_early_MOTOR_collapsed", 
  "ICPS_early_OCC_collapsed"
)

# 2. Define the static right-hand side of your model formula
predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}


Fitting model for power_early...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1591.1

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.7974 -0.5342 -0.0119  0.5582  3.8864 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1477   0.3843  
 Residual             0.2637   0.5135  
Number of obs: 855, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)       0.015195   0.044902 226.905048   0.338 0.735374    
acc1             -0.735293   0.017612 623.341398 -41.750  < 2e-16 ***
soc1              0.013586   0.017914 653.118858   0.758 0.448483    
age_m             0.167646   0.030936 228.829763   5.419 1.51e-07 ***
sex1             -0.120474   0.030973 226.571050  -3.890 0.000132 ***
dp_inperson1     -0.003830   0.044947 226.148124  -0.085 0.932172    
acc1:soc1        -0.010934   0.

Computing profile confidence intervals ...



Fitting model for ITPS_early...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 2349.8

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.5018 -0.5728 -0.0680  0.4644  3.7161 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2150   0.4636  
 Residual             0.7257   0.8519  
Number of obs: 851, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)      -0.006729   0.061703 221.663336  -0.109 0.913254    
acc1             -0.184696   0.029276 617.820849  -6.309 5.36e-10 ***
soc1              0.004533   0.029643 655.164214   0.153 0.878502    
age_m             0.144965   0.042530 223.282828   3.409 0.000775 ***
sex1              0.085859   0.042514 220.375038   2.020 0.044639 *  
dp_inperson1      0.003758   0.061735 220.801215   0.061 0.951512    
acc1:soc1         0.019765   0.0

Computing profile confidence intervals ...



Fitting model for ICPS_early_DLPFC_collapsed...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 2149.3

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.0122 -0.5189 -0.0542  0.4275  4.0678 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1731   0.4160  
 Residual             0.5835   0.7639  
Number of obs: 844, groups:  sub, 233

Fixed effects:
                 Estimate Std. Error        df t value Pr(>|t|)    
(Intercept)      -0.02170    0.05570 227.49610  -0.389  0.69727    
acc1             -0.45487    0.02637 614.83537 -17.248  < 2e-16 ***
soc1             -0.01685    0.02670 650.94491  -0.631  0.52812    
age_m             0.19862    0.03833 227.20353   5.182 4.83e-07 ***
sex1              0.02145    0.03824 222.54374   0.561  0.57535    
dp_inperson1      0.03588    0.05573 226.55085   0.644  0.52033    
acc1:soc1         0.03369    0

Computing profile confidence intervals ...



Fitting model for ICPS_early_MOTOR_collapsed...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1985.2

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.6643 -0.5333 -0.0319  0.4864  3.8045 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1552   0.3939  
 Residual             0.4679   0.6840  
Number of obs: 847, groups:  sub, 232

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)      -0.023507   0.051072 216.873408  -0.460    0.646    
acc1             -0.580142   0.023577 611.410111 -24.606  < 2e-16 ***
soc1              0.013105   0.023874 645.775769   0.549    0.583    
age_m             0.183365   0.035260 218.539750   5.200 4.57e-07 ***
sex1             -0.015308   0.035241 215.920575  -0.434    0.664    
dp_inperson1      0.048767   0.051106 216.016256   0.954    0.341    
acc1:soc1       

Computing profile confidence intervals ...



Fitting model for ICPS_early_OCC_collapsed...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 2211.7

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.9056 -0.5639  0.0057  0.5162  3.9995 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2189   0.4679  
 Residual             0.6229   0.7892  
Number of obs: 839, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)      1.920e-02  5.992e-02  2.183e+02   0.320   0.7490    
acc1            -3.981e-01  2.737e-02  6.094e+02 -14.544   <2e-16 ***
soc1             4.762e-03  2.772e-02  6.416e+02   0.172   0.8637    
age_m            6.603e-02  4.144e-02  2.219e+02   1.593   0.1125    
sex1            -6.884e-02  4.143e-02  2.192e+02  -1.661   0.0981 .  
dp_inperson1    -1.237e-02  5.995e-02  2.175e+02  -0.206   0.8368    
acc1:soc1        3

Computing profile confidence intervals ...



In [5]:
outcome <- "ICPS_early_DLPFC_collapsed"
predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)
print(summary(model))

Fitting model for ICPS_early_DLPFC_collapsed...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 2146.3

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.0190 -0.5262 -0.0515  0.4321  4.0769 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1712   0.4138  
 Residual             0.5821   0.7629  
Number of obs: 844, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)      -0.020438   0.055518 227.364591  -0.368 0.713121    
acc1             -0.455084   0.026339 614.763581 -17.278  < 2e-16 ***
soc1             -0.016680   0.026662 650.943589  -0.626 0.531799    
age_m             0.204191   0.038211 226.439138   5.344 2.22e-07 ***
sex1              0.023088   0.038110 222.411495   0.606 0.545254    
dp_inperson1      0.034419   0.055547 226.426992   0.620 0.536120    
acc1:soc1       

### Plot acc*age ICPS

In [27]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

for (outcome in c("power_early", "ITPS_early")) {

    predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
    f <- as.formula(paste(outcome, "~", predictors))
    cat(sprintf("Fitting model for %s...\n", outcome))
    
    model <- lmer(f, data = tf_data)
    # print(summary(model))
    x_label = "Age"    
    if (outcome == "power_early") {
        y_label = "Power"
        label = "power"
    } else if (outcome == "ITPS_early") {
        y_label = "ITPS"
        label = "ITPS"
    }
    
    points = FALSE
    if (points == FALSE) {
        png(file=sprintf("%s/%s_stats_no_points.png", pic_path, label), width=10, height=4, units="in", res=600)
    } else {
        png(file=sprintf("%s/%s_stats.png", pic_path, label), width=10, height=4, units="in", res=600)
    }
    interact_plot(model,
                  pred = "age_m",
                  modx = "acc",
                  interval = 1,
                  y.label = y_label,
                  x.label = x_label,
                  legend.main = "Accuracy",
                  colors = c("red", "blue"),
                  plot.points = points) +
    theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
      scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
    theme(
      text = element_text(size = 14),                              # Change the base font size
      axis.title = element_text(size = 16),                        # Change the axis title font size
      axis.text = element_text(size = 14),                         # Change the axis text font size
      legend.text = element_text(size = 14),                       # Change the legend text font size
      legend.title = element_text(size = 14), 
      plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
    )
    dev.off()
}

Fitting model for power_early...
[1] "power"
Fitting model for ITPS_early...
[1] "ITPS"


In [48]:
for (outcome in c("ICPS_early_DLPFC_collapsed", "ICPS_early_MOTOR_collapsed")) {

    predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
    f <- as.formula(paste(outcome, "~", predictors))
    cat(sprintf("Fitting model for %s...\n", outcome))
    
    model <- lmer(f, data = tf_data)
    
    x_label = "Age"    
    if (outcome == "ICPS_early_DLPFC_collapsed") {
        y_label = "Frontolateral ICPS"
        label = "ICPS_DLPFC"
    } else if (outcome == "ICPS_early_MOTOR_collapsed") {
        y_label = "Midlateral ICPS"
        label = "ICPS_MOTOR"
    }
    
    points = TRUE
    file_name <- if (!points) sprintf("%s/%s_stats_no_points.png", pic_path, label) else sprintf("%s/%s_stats.png", pic_path, label)

    # Assign plot to an object
    p <- interact_plot(model,
                  pred = "age_m",
                  modx = "acc",
                  interval = 1,
                  y.label = y_label,
                  x.label = x_label,
                  legend.main = "Accuracy",
                  colors = c("red", "blue"),
                  plot.points = points) +
        theme_minimal() +  
        scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
        scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
        theme(
          text = element_text(size = 14),                                
          axis.title = element_text(size = 16),                          
          axis.text = element_text(size = 14),                           
          legend.text = element_text(size = 14),                         
          legend.title = element_text(size = 14), 
            legend.position = "none",
          plot.title = element_text(size = 18, face = "bold")            
        )
    
    # ggsave handles printing and rendering automatically
    ggsave(filename = file_name, plot = p, width = 5, height = 4, units = "in", dpi = 600)
}

Fitting model for ICPS_early_DLPFC_collapsed...
Fitting model for ICPS_early_MOTOR_collapsed...


In [45]:
pic_path

[1] "/home/data/NDClab/analyses/thrive-theta-ddm//derivatives/figs/s1_r1"

In [30]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

# outcome = "power_early"
outcome = "ITPS_early"

predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))
    
model <- lmer(f, data = tf_data)
# print(summary(model))
x_label = "Age"    
if (outcome == "power_early") {
    y_label = "Power"
    label = "power"
} else if (outcome == "ITPS_early") {
    y_label = "ITPS"
    label = "ITPS"
}

points = FALSE
if (points == FALSE) {
    png(file=sprintf("%s/%s_stats_no_points.png", pic_path, label), width=10, height=4, units="in", res=600)
} else {
    png(file=sprintf("%s/%s_stats.png", pic_path, label), width=10, height=4, units="in", res=600)
}
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = points) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14), 
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

Fitting model for ITPS_early...


agg_record_1106239991 
                    2

In [22]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

# outcome <- "ICPS_early_DLPFC_collapsed"
# outcome <- "ICPS_early_MOTOR_collapsed"
outcome <- "ITPS_early"
predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)
print(summary(model))

# y_label = "Frontolateral ICPS"
# y_label = "Midlateral ICPS"
# y_label = "Power"
y_label = "ITPS"
x_label = "Age"
# label = "ICPS_DLPFC"
# label = "ICPS_MOTOR"
# label = "power"
label = "ITPS"

points = FALSE
if (points == FALSE) {
    png(file=sprintf("%s/%s_stats_no_points.png", pic_path, label), width=5, height=4, units="in", res=600)
} else {
    png(file=sprintf("%s/%s_stats.png", pic_path, label), width=5, height=4, units="in", res=600)
}
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = points) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14), 
        legend.position = "none", # Change the legend title font size# Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

Fitting model for power_early...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1590.4

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.7967 -0.5339 -0.0085  0.5582  3.8812 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1479   0.3846  
 Residual             0.2633   0.5132  
Number of obs: 855, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)       0.015636   0.044918 226.874565   0.348 0.728087    
acc1             -0.735354   0.017600 623.311749 -41.780  < 2e-16 ***
soc1              0.013577   0.017903 653.072684   0.758 0.448495    
age_m             0.167464   0.030969 228.434590   5.407 1.61e-07 ***
sex1             -0.119092   0.030985 226.577448  -3.844 0.000158 ***
dp_inperson1     -0.004465   0.044967 226.122430  -0.099 0.920986    
acc1:soc1        -0.010976   0.

agg_record_1475537867 
                    2

### Plot acc*age*soc ICPS

In [25]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

# outcome <- "ICPS_early_DLPFC_collapsed"
outcome <- "ICPS_early_OCC_collapsed"
predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)
print(summary(model))

# y_label = "Frontolateral ICPS"
y_label = "Posterolateral ICPS"
x_label = "Age"
# label = "ICPS_DLPFC"
label = "ICPS_OCC"

points = TRUE
if (points == FALSE) {
    png(file=sprintf("%s/%s_stats_no_points.png", pic_path, label), width=9, height=4, units="in", res=600)
} else {
    png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=4, units="in", res=600)
}
cond_labels = c("Non-social", "Social")

interact_plot(model,
              pred = "age_m",
              modx = "acc",
              mod2="soc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              colors = c("red", "blue"),
              legend.main = "Accuracy",
              mod2.labels = cond_labels,
              plot.points = points) +
  theme_minimal() +
  theme(
    text = element_text(size = 20),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    legend.text = element_text(size = 14),                       # Change the legend text font size
    legend.title = element_text(size = 14),                     # Change the legend title font size
  legenbbd.position = "none",
    plot.title = element_text(size = 20, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-1, 1, by = 1)) + 
  scale_linetype_manual(values = c("Error" = "solid", "Correct" = "dashed"))
dev.off()

Fitting model for ICPS_early_OCC_collapsed...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 2211.7

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.9056 -0.5639  0.0057  0.5162  3.9995 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2189   0.4679  
 Residual             0.6229   0.7892  
Number of obs: 839, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)      1.920e-02  5.992e-02  2.183e+02   0.320   0.7490    
acc1            -3.981e-01  2.737e-02  6.094e+02 -14.544   <2e-16 ***
soc1             4.762e-03  2.772e-02  6.416e+02   0.172   0.8637    
age_m            6.603e-02  4.144e-02  2.219e+02   1.593   0.1125    
sex1            -6.884e-02  4.143e-02  2.192e+02  -1.661   0.0981 .  
dp_inperson1    -1.237e-02  5.995e-02  2.175e+02  -0.206   0.8368    
acc1:soc1        3

Scale for linetype is already present.
Adding another scale for linetype, which will replace the existing scale.


agg_record_601917852 
                   2

## Posthocs

In [13]:
# outcome <- "ICPS_early_DLPFC_collapsed"
# outcome <- "ICPS_early_MOTOR_collapsed"
# outcome <- "power_early"
outcome <- "ITPS_early"
predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))
model <- lmer(f, data = tf_data)
simple_slopes <- emtrends(model, ~ acc, var = "age_m")
summary(model)

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE), adjust = 'fdr')

# Calculate the difference between the slopes across levels of 'acc'
slope_diffs <- pairs(simple_slopes)

# View the differences and significance tests
summary(slope_diffs, infer = c(TRUE, TRUE))

library(broom)
library(knitr)

# tidy() converts the objects into clean tibbles
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)
tidy_diffs <- tidy(slope_diffs, conf.int = TRUE)

cat("\n--- Simple Slopes ---\n")
print(kable(tidy_slopes, digits = 3, format = "simple"))

cat("\n--- Slope Differences (e.g., Error - Correct) ---\n")
print(kable(tidy_diffs, digits = 3, format = "simple"))

Fitting model for ITPS_early...


Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 2349.8

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.5018 -0.5728 -0.0680  0.4644  3.7161 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2150   0.4636  
 Residual             0.7257   0.8519  
Number of obs: 851, groups:  sub, 233

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)      -0.006729   0.061703 221.663336  -0.109 0.913254    
acc1             -0.184696   0.029276 617.820849  -6.309 5.36e-10 ***
soc1              0.004533   0.029643 655.164214   0.153 0.878502    
age_m             0.144965   0.042530 223.282828   3.409 0.000775 ***
sex1              0.085859   0.042514 220.375038   2.020 0.044639 *  
dp_inperson1      0.003758   0.061735 220.801215   0.061 0.951512    
acc1:soc1         0.019765   0.029263 616.515781   0.675 0.49965

,acc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,0.25375843,0.05192320,452.9087,0.13698941,0.3705274,4.8871880,2.84420e-06
2,Correct,0.03617154,0.05140147,445.8832,-0.07943025,0.1517733,0.7037063,4.81983e-01


,contrast,estimate,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error - Correct,0.2175869,0.05866001,617.733,0.1023897,0.3327841,3.709288,0.0002265663



--- Simple Slopes ---


acc        age_m.trend   std.error        df   conf.low   conf.high   statistic   p.value
--------  ------------  ----------  --------  ---------  ----------  ----------  --------
Error            0.254       0.052   452.909      0.152       0.356       4.887     0.000
Correct          0.036       0.051   445.883     -0.065       0.137       0.704     0.482

--- Slope Differences (e.g., Error - Correct) ---


term   contrast           null.value   estimate   std.error        df   conf.low   conf.high   statistic   p.value
-----  ----------------  -----------  ---------  ----------  --------  ---------  ----------  ----------  --------
acc    Error - Correct             0      0.218       0.059   617.733      0.102       0.333       3.709         0


In [11]:
outcome <- "ICPS_early_OCC_collapsed"
predictors <- "acc * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))
model <- lmer(f, data = tf_data)


# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc | soc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr'
       )

diff_slopes <- contrast(simple_slopes, method = "revpairwise", by = "soc")

# View differences and significance tests
summary(diff_slopes, infer = c(TRUE, TRUE),
        adjust='fdr'
       )

diff_slopes <- contrast(simple_slopes, method = "revpairwise", by = "acc")

# View differences and significance tests
summary(diff_slopes, infer = c(TRUE, TRUE),
        adjust='fdr'
       )

Fitting model for ICPS_early_OCC_collapsed...


Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed



,acc,soc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,NS,0.16695676,0.06395558,730.4842,0.02331043,0.3106031,2.6105112,0.01845207
2,Correct,NS,0.05279729,0.06299284,722.1399,-0.08869007,0.1942846,0.8381475,0.40222519
3,Error,S,-0.03512447,0.06304974,722.3942,-0.17673953,0.1064906,-0.5570914,0.57763755
4,Correct,S,0.07949171,0.06272149,718.0187,-0.06138787,0.2203713,1.2673760,0.41086353


,contrast,soc,estimate,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Correct - Error,NS,-0.1141595,0.07792089,607.8141,-0.26718633,0.0388674,-1.465069,0.1434192
2,Correct - Error,S,0.1146162,0.07698563,605.5305,-0.03657508,0.2658074,1.488800,0.1370607


,contrast,acc,estimate,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,S - NS,Error,-0.20208122,0.07858569,629.6284,-0.3564030,-0.04775945,-2.5714761,0.01035495
2,S - NS,Correct,0.02669442,0.07759278,626.8229,-0.1256788,0.17906769,0.3440323,0.73093734


### Plot

In [56]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Power"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=8, height=5, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
# scale_color_manual(values = c("Error" = "red", "Correct" = "blue")) +
# scale_fill_manual(values = c("Error" = "red", "Correct" = "blue")) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14),                     # Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

agg_record_561350890 
                   2

In [19]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,acc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,0.25582531,0.05190172,452.2180,0.13910400,0.3725466,4.9290332,2.323960e-06
2,Correct,0.03787141,0.05139765,445.4925,-0.07772214,0.1534650,0.7368314,4.616127e-01




acc        age_m.trend   std.error        df   conf.low   conf.high   statistic   p.value
--------  ------------  ----------  --------  ---------  ----------  ----------  --------
Error            0.256       0.052   452.218      0.154       0.358       4.929     0.000
Correct          0.038       0.051   445.493     -0.063       0.139       0.737     0.462

### Plot

In [19]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "ITPS"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=8, height=5, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
# scale_color_manual(values = c("Error" = "red", "Correct" = "blue")) +
# scale_fill_manual(values = c("Error" = "red", "Correct" = "blue")) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14),                     # Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

agg_record_1148129182 
                    2

In [8]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,acc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,0.2987916,0.04667660,457.4166,0.193824943,0.4037582,6.401313,7.644664e-10
2,Correct,0.1095913,0.04630009,452.3003,0.005467496,0.2137151,2.366978,1.835419e-02


In [12]:
library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")



acc        age_m.trend   std.error        df   conf.low   conf.high   statistic   p.value
--------  ------------  ----------  --------  ---------  ----------  ----------  --------
Error            0.299       0.047   457.417      0.207       0.391       6.401     0.000
Correct          0.110       0.046   452.300      0.019       0.201       2.367     0.018

### Plot

Fitting model for ICPS_early_MOTOR_collapsed...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1987

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.6635 -0.5330 -0.0267  0.4874  3.7932 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1558   0.3948  
 Residual             0.4687   0.6846  
Number of obs: 847, groups:  sub, 232

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)     -2.305e-02  5.116e-02  2.168e+02  -0.451    0.653    
acc1            -5.802e-01  2.360e-02  6.114e+02 -24.588  < 2e-16 ***
soc1             1.306e-02  2.390e-02  6.457e+02   0.547    0.585    
age_m            1.814e-01  3.534e-02  2.180e+02   5.133 6.30e-07 ***
sex1            -1.381e-02  3.530e-02  2.159e+02  -0.391    0.696    
dp_inperson1     4.824e-02  5.120e-02  2.160e+02   0.942    0.347    
acc1:soc1       -1

agg_record_1818158400 
                    2

In [15]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

simple_slopes <- emtrends(model, ~ acc, var = "age_m")

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE),
        adjust='fdr',
       )

library(broom)
library(knitr)

# tidy() converts the object into a clean tibble
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)

kable(tidy_slopes, digits = 3, format = "simple")

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,acc,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Error,0.28769250,0.04265601,431.2919,0.19174823,0.3836368,6.744477,9.905076e-11
2,Correct,0.07508435,0.04237804,425.7763,-0.02023902,0.1704077,1.771775,7.714709e-02




acc        age_m.trend   std.error        df   conf.low   conf.high   statistic   p.value
--------  ------------  ----------  --------  ---------  ----------  ----------  --------
Error            0.288       0.043   431.292      0.204       0.372       6.744     0.000
Correct          0.075       0.042   425.776     -0.008       0.158       1.772     0.077

### Plot

In [48]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Midlateral ICPS"
x_label = "Age"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=5, height=4, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              legend.main = "Accuracy",
              colors = c("red", "blue"),
              plot.points = T) +
theme_minimal() +  scale_y_continuous(breaks = seq(-3, 4, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) +
theme(
  text = element_text(size = 14),                              # Change the base font size
  axis.title = element_text(size = 16),                        # Change the axis title font size
  axis.text = element_text(size = 14),                         # Change the axis text font size
  legend.text = element_text(size = 14),                       # Change the legend text font size
  legend.title = element_text(size = 14),
    legend.position = "none", # Change the legend title font size
  plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
)
dev.off()

agg_record_909403265 
                   2

### Plot

In [67]:
# test(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")
# confint(emtrends(model, pairwise ~ acc | age_m, var="age_m"), adjust="fdr")

y_label = "Posterolateral ICPS"
x_label = "Age"
cond_labels = c("Non-social", "Social")

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=8, height=5, units="in", res=600)
interact_plot(model,
              pred = "age_m",
              modx = "acc",
              mod2="soc",
              interval = 1,
              y.label = y_label,
              x.label = x_label,
              colors = c("red", "blue"),
              legend.main = "Accuracy",
              mod2.labels = cond_labels,
              plot.points = T) +
  theme_minimal() +
  theme(
    text = element_text(size = 20),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    legend.text = element_text(size = 14),                       # Change the legend text font size
    legend.title = element_text(size = 14),                     # Change the legend title font size
    plot.title = element_text(size = 20, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-1, 1, by = 1))
  # scale_linetype_manual(values = c("solid", "solid"))
dev.off()

agg_record_1006651352 
                    2

# DDM

## Load data

In [3]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}
# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'reversed_ratio_diff',
    'a_diff',
    'p_diff',
    'ter_diff',
    'ICPS_early_DLPFC_diff_collapsed',
    'ICPS_early_MOTOR_diff_collapsed',
    'ICPS_early_OCC_diff_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_20_05_2026_14_17_15.csv"


## Full models (all three ICPS)

In [27]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'reversed_ratio_diff',
    'a_diff'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_full")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

Fitting model for reversed_ratio_diff...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1026.6

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.37183 -0.61550  0.02435  0.54938  2.59486 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1092   0.3305  
 Residual             0.8994   0.9484  
Number of obs: 344, groups:  sub, 207

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.064091   0.083292 193.589235
ICPS_early_DLPFC_diff_collapsed             -0.025507   0.072040 325.310002
soc1                                        -0.052215   0.053405 167.939339
age_m                                       -0.010067   0.058991 177.220297
ICPS_early_MOTOR_diff_collapsed             -0.013091   0.072757 303.775830
ICPS_early_OCC_diff_collapsed                0.


Correlation matrix not shown by default, as p = 18 > 12.
Use print(summary(model), correlation=TRUE)  or
    vcov(summary(model))        if you need it


Computing profile confidence intervals ...



Fitting model for a_diff...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1003.6

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.77888 -0.63006 -0.04662  0.66071  2.49488 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.05307  0.2304  
 Residual             0.91692  0.9576  
Number of obs: 340, groups:  sub, 205

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                  0.024692   0.081735 190.919930
ICPS_early_DLPFC_diff_collapsed              0.025023   0.071408 320.227798
soc1                                         0.009873   0.053928 167.365622
age_m                                        0.109563   0.056955 170.906716
ICPS_early_MOTOR_diff_collapsed             -0.023575   0.071418 292.261779
ICPS_early_OCC_diff_collapsed                0.105861   0.06


Correlation matrix not shown by default, as p = 18 > 12.
Use print(summary(model), correlation=TRUE)  or
    vcov(summary(model))        if you need it


Computing profile confidence intervals ...



## DLPFC-only

In [28]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'reversed_ratio_diff',
    'a_diff'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_dlpfc")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

Fitting model for reversed_ratio_diff...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1060.8

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.54559 -0.56871  0.05991  0.53215  2.53303 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1729   0.4158  
 Residual             0.8701   0.9328  
Number of obs: 360, groups:  sub, 213

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.043182   0.084212 212.667298
ICPS_early_DLPFC_diff_collapsed              0.043160   0.060176 347.367598
soc1                                        -0.066215   0.050496 179.598112
age_m                                       -0.006233   0.058716 193.689733
sex1                                        -0.043713   0.057405 187.738585
dp_inperson1                                 0.

Computing profile confidence intervals ...



Fitting model for a_diff...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1031.1

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.73804 -0.62048 -0.03077  0.70821  2.55646 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.04442  0.2108  
 Residual             0.93443  0.9667  
Number of obs: 356, groups:  sub, 211

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.004984   0.079923 210.726551
ICPS_early_DLPFC_diff_collapsed              0.055352   0.058667 335.820531
soc1                                        -0.007718   0.052003 183.192611
age_m                                        0.100340   0.054712 186.874245
sex1                                         0.114781   0.053511 183.490986
dp_inperson1                                 0.023785   0.07

Computing profile confidence intervals ...



## MOTOR-only

In [29]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'reversed_ratio_diff',
    'a_diff'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_MOTOR_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_motor")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

Fitting model for reversed_ratio_diff...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1071

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.43645 -0.60521  0.02817  0.57776  2.56926 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1570   0.3962  
 Residual             0.8622   0.9285  
Number of obs: 366, groups:  sub, 214

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                -5.863e-02  8.167e-02  2.085e+02
ICPS_early_MOTOR_diff_collapsed             4.130e-02  5.889e-02  3.179e+02
soc1                                       -5.753e-02  5.007e-02  1.859e+02
age_m                                      -2.558e-02  5.729e-02  1.983e+02
sex1                                       -3.197e-02  5.626e-02  1.933e+02
dp_inperson1                                5.572

Computing profile confidence intervals ...



Fitting model for a_diff...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1056.5

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.74210 -0.64044 -0.01069  0.64631  2.64951 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.04715  0.2171  
 Residual             0.95402  0.9767  
Number of obs: 362, groups:  sub, 212

Fixed effects:
                                            Estimate Std. Error        df
(Intercept)                                 -0.01744    0.07943 203.46064
ICPS_early_MOTOR_diff_collapsed              0.02159    0.05779 295.60353
soc1                                        -0.03488    0.05248 186.63929
age_m                                        0.10563    0.05488 189.86913
sex1                                         0.12541    0.05393 187.19901
dp_inperson1                                 0.04506    0.07856 201.0965

Computing profile confidence intervals ...



## OCC-only

In [30]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'reversed_ratio_diff',
    'a_diff'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_occ")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

Fitting model for reversed_ratio_diff...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1051

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.43707 -0.62922  0.04827  0.58080  2.58213 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1392   0.3731  
 Residual             0.8308   0.9115  
Number of obs: 365, groups:  sub, 213

Fixed effects:
                                           Estimate Std. Error         df
(Intercept)                               -0.030018   0.077966 194.165349
ICPS_early_OCC_diff_collapsed              0.166767   0.056436 326.076785
soc1                                      -0.049289   0.048676 176.639563
age_m                                      0.017070   0.054996 183.882063
sex1                                      -0.027496   0.054755 184.911868
dp_inperson1                               0.056347   0.07808

Computing profile confidence intervals ...



Fitting model for a_diff...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1042.5

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.74571 -0.64928 -0.03647  0.64513  2.59248 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.02778  0.1667  
 Residual             0.94171  0.9704  
Number of obs: 361, groups:  sub, 211

Fixed effects:
                                           Estimate Std. Error         df
(Intercept)                               -0.038379   0.076121 191.092595
ICPS_early_OCC_diff_collapsed              0.071381   0.055942 306.833626
soc1                                      -0.005368   0.051582 179.451656
age_m                                      0.102914   0.052810 177.134298
sex1                                       0.144186   0.052677 180.072382
dp_inperson1                               0.065971   0.076305 192.13184

Computing profile confidence intervals ...



In [12]:
outcome <- "reversed_ratio_diff"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "ratio_reversed"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
g <- ggplot(data = tf_data, aes(x = ICPS_early_OCC_diff_collapsed, y = reversed_ratio_diff))
g + geom_smooth(method = "lm") + 
geom_point() + 
labs(y=plot_label, x="Midfrontal-Posterolateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1))
dev.off()

Fitting model for reversed_ratio_diff...


`geom_smooth()` using formula = 'y ~ x'
Warning message:
"Removed 77 rows containing non-finite outside the scale range
(`stat_smooth()`)."
Warning message:
"Removed 77 rows containing missing values or values outside the scale range
(`geom_point()`)."


agg_record_381156843 
                   2

In [4]:
outcome <- "a_diff"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "a_diff"
plot_label <- bquote(atop("Response Caution", alpha ~ "(Post-error - Post-correct)"))

png(file=sprintf("%s/%s_stats_all_three.png", pic_path, label), width=9, height=5, units="in", res=600)
interact_plot(model, pred = "ICPS_early_DLPFC_diff_collapsed", modx = "age_m", interval = 1, dodge.width = 0.2,
              y.label = plot_label,
              x.label = "Midfrontal-Frontolateral ICPS (Error - Correct)",
              legend.main = "Age",
              plot.points = T) + 
  theme_minimal() +
  theme(
    text = element_text(size = 14), # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    # legend.text = element_text(size = 14),                       # Change the legend text font size
    # legend.title = element_text(size = 14),                     # Change the legend title font size
    plot.title = element_text(size = 18, face = "bold")         # Change the plot title font size and make it bold
  ) + theme(legend.position = "none") +
  scale_y_continuous(breaks = seq(-3, 3, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1)) + theme(legend.position = "bottom")
dev.off()

Fitting model for a_diff...


agg_record_1470476795 
                    2

In [5]:
library(emmeans)

# 1. Directly specify the standardized values (-1 SD, Mean, +1 SD)
age_points <- list(age_m = c(-1, 0, 1))

# 2. Compute the simple slopes
slopes <- emtrends(model, 
                   specs = ~ age_m, 
                   var = "ICPS_early_DLPFC_diff_collapsed", 
                   at = age_points)

# 3. View the slopes with 95% Confidence Intervals and p-values
summary(slopes, infer = c(TRUE, TRUE))

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,age_m,ICPS_early_DLPFC_diff_collapsed.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,-1,0.17390826,0.11321865,321.9372,-0.0488336,0.39665011,1.5360389,0.1255110
2,0,0.02348432,0.07126169,320.1717,-0.1167160,0.16368463,0.3295504,0.7419552
3,1,-0.12693962,0.09251097,316.7672,-0.3089532,0.05507398,-1.3721574,0.1709853


In [6]:
library(emmeans)

# 1. Compute the slopes at the standardized age points
age_points <- list(age_m = c(-1, 0, 1))
slopes <- emtrends(model, 
                   specs = ~ age_m, 
                   var = "ICPS_early_DLPFC_diff_collapsed", 
                   at = age_points)

# 2. Compute paired comparisons between the slopes
pairwise_diffs <- pairs(slopes)

# 3. View the results
summary(pairwise_diffs, infer = c(TRUE, TRUE))

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

NOTE: Results may be misleading due to involvement in interactions



,contrast,estimate,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,(age_m-1) - age_m0,0.1504239,0.07490089,321.3321,-0.02593965,0.3267875,2.008306,0.1118253
2,(age_m-1) - age_m1,0.3008479,0.14980179,321.3321,-0.05187930,0.6535751,2.008306,0.1118253
3,age_m0 - age_m1,0.1504239,0.07490089,321.3321,-0.02593965,0.3267875,2.008306,0.1118253


# Updated raw behav measures

## Load data

In [39]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}
# it doesnt matter 0 or 1 because the models below use difference scores; because of that we need to get rid of duplicates (diff scores are the same for acc=0 and acc=1)
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'pea',
    'peri_rt',
    'pes',
    'ICPS_early_DLPFC_diff_collapsed',
    'ICPS_early_MOTOR_diff_collapsed',
    'ICPS_early_OCC_diff_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))
tf_data <- subset(tf_data, tf_data$acc == 1)

tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
# tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

# contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_20_05_2026_14_17_15.csv"


## Full models (all three ICPS)

In [32]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'pea',
    'peri_rt',
    'pes'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_full")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

Fitting model for pea...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1083.4

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.5138 -0.5220  0.0657  0.5894  2.4820 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1011   0.3180  
 Residual             0.8363   0.9145  
Number of obs: 372, groups:  sub, 216

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.080826   0.077556 199.536059
ICPS_early_DLPFC_diff_collapsed             -0.027385   0.059870 351.818188
soc1                                        -0.031665   0.049663 185.655190
age_m                                        0.155425   0.054091 194.712600
ICPS_early_MOTOR_diff_collapsed              0.020891   0.063910 333.853160
ICPS_early_OCC_diff_collapsed                0.108609   0.054298 346.9155


Correlation matrix not shown by default, as p = 18 > 12.
Use print(summary(model), correlation=TRUE)  or
    vcov(summary(model))        if you need it


Computing profile confidence intervals ...



Fitting model for peri_rt...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1087.7

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.3340 -0.5833 -0.0031  0.5657  2.9397 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.08269  0.2876  
 Residual             0.87240  0.9340  
Number of obs: 371, groups:  sub, 215

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 3.325e-02  7.818e-02  2.133e+02
ICPS_early_DLPFC_diff_collapsed             1.062e-01  6.149e-02  3.507e+02
soc1                                        9.943e-02  5.072e-02  1.984e+02
age_m                                      -1.043e-02  5.451e-02  2.063e+02
ICPS_early_MOTOR_diff_collapsed            -1.205e-01  6.497e-02  3.357e+02
ICPS_early_OCC_diff_collapsed               1.525e-02  5.501e-02  3.4


Correlation matrix not shown by default, as p = 18 > 12.
Use print(summary(model), correlation=TRUE)  or
    vcov(summary(model))        if you need it


Computing profile confidence intervals ...



Fitting model for pes...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1061

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.79158 -0.55191  0.02932  0.58706  2.67259 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2343   0.4840  
 Residual             0.6674   0.8169  
Number of obs: 372, groups:  sub, 217

Fixed effects:
                                            Estimate Std. Error        df
(Intercept)                                 -0.01217    0.07986 212.50677
ICPS_early_DLPFC_diff_collapsed             -0.12313    0.05793 353.27491
soc1                                        -0.03549    0.04504 185.46893
age_m                                        0.03955    0.05565 205.78053
ICPS_early_MOTOR_diff_collapsed              0.14331    0.06306 349.52283
ICPS_early_OCC_diff_collapsed               -0.03560    0.05296 353.94305
sex


Correlation matrix not shown by default, as p = 18 > 12.
Use print(summary(model), correlation=TRUE)  or
    vcov(summary(model))        if you need it


Computing profile confidence intervals ...



In [6]:
outcome <- "pea"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "pea"
plot_label <- "Post-Error Accuracy"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
g <- ggplot(data = tf_data, aes(x = ICPS_early_OCC_diff_collapsed, y = pea))
g + geom_smooth(method = "lm") + 
geom_point() + 
labs(y=plot_label, x="Midfrontal-Posterolateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1))
dev.off()

Fitting model for pea...


`geom_smooth()` using formula = 'y ~ x'
Warning message:
"Removed 52 rows containing non-finite outside the scale range
(`stat_smooth()`)."
Warning message:
"Removed 52 rows containing missing values or values outside the scale range
(`geom_point()`)."


agg_record_32101405 
                  2

In [7]:
outcome <- "pes"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "pes_fl"
plot_label <- "Post-Error Slowing"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
g <- ggplot(data = tf_data, aes(x = ICPS_early_DLPFC_diff_collapsed, y = pes))
g + geom_smooth(method = "lm") + 
geom_point() + 
labs(y=plot_label, x="Midfrontal-Frontolateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1))
dev.off()

Fitting model for pes...


`geom_smooth()` using formula = 'y ~ x'
Warning message:
"Removed 51 rows containing non-finite outside the scale range
(`stat_smooth()`)."
Warning message:
"Removed 51 rows containing missing values or values outside the scale range
(`geom_point()`)."


agg_record_1400445359 
                    2

In [8]:
outcome <- "pes"
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
cat(sprintf("Fitting model for %s...\n", outcome))

model <- lmer(f, data = tf_data)

label <- "pes_ml"
plot_label <- "Post-Error Slowing"

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)
g <- ggplot(data = tf_data, aes(x = ICPS_early_MOTOR_diff_collapsed, y = pes))
g + geom_smooth(method = "lm") + 
geom_point() + 
labs(y=plot_label, x="Midfrontal-Midlateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),                              # Change the base font size
    axis.title = element_text(size = 16),                        # Change the axis title font size
    axis.text = element_text(size = 14),                         # Change the axis text font size
    plot.title = element_text(size = 18, face = "bold")          # Change the plot title font size and make it bold
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-3, 3, by = 1))
dev.off()

Fitting model for pes...


`geom_smooth()` using formula = 'y ~ x'
Warning message:
"Removed 43 rows containing non-finite outside the scale range
(`stat_smooth()`)."
Warning message:
"Removed 43 rows containing missing values or values outside the scale range
(`geom_point()`)."


agg_record_1144226340 
                    2

## DLPFC-only

In [33]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'pea',
    'peri_rt',
    'pes'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_DLPFC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_dlpfc")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

Fitting model for pea...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1149.4

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.13875 -0.52886  0.02841  0.59887  2.73741 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1491   0.3861  
 Residual             0.8280   0.9099  
Number of obs: 399, groups:  sub, 224

Fixed effects:
                                            Estimate Std. Error        df
(Intercept)                                 -0.07537    0.07813 216.45333
ICPS_early_DLPFC_diff_collapsed              0.02562    0.05143 385.77638
soc1                                        -0.02959    0.04658 194.56109
age_m                                        0.17523    0.05352 205.12450
sex1                                         0.03784    0.05270 197.36425
dp_inperson1                                 0.08373    0.07777 215.77858
I

Computing profile confidence intervals ...



Fitting model for peri_rt...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1163.5

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.3136 -0.5188  0.0066  0.5561  2.9247 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.0834   0.2888  
 Residual             0.9298   0.9643  
Number of obs: 398, groups:  sub, 225

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                  0.016284   0.078354 226.822501
ICPS_early_DLPFC_diff_collapsed              0.034730   0.052897 379.312157
soc1                                         0.111144   0.049307 205.521374
age_m                                       -0.024481   0.053451 213.656815
sex1                                        -0.006871   0.052280 204.080191
dp_inperson1                                -0.024985   0.077836 225.

Computing profile confidence intervals ...



Fitting model for pes...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1116.2

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.07540 -0.56390  0.01173  0.58561  2.62878 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2677   0.5174  
 Residual             0.6553   0.8095  
Number of obs: 399, groups:  sub, 224

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.019974   0.079550 229.191581
ICPS_early_DLPFC_diff_collapsed             -0.073698   0.049648 388.455456
soc1                                        -0.029574   0.041787 197.697981
age_m                                        0.065066   0.054412 215.610790
sex1                                         0.235158   0.053781 208.083799
dp_inperson1                                 0.051261   0.07929

Computing profile confidence intervals ...



## MOTOR-only

In [34]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'pea',
    'peri_rt',
    'pes'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_MOTOR_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_motor")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

Fitting model for pea...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1166.5

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.2025 -0.5311  0.0344  0.6099  2.7758 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1612   0.4015  
 Residual             0.8042   0.8968  
Number of obs: 407, groups:  sub, 225

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.056692   0.076119 214.736141
ICPS_early_MOTOR_diff_collapsed              0.070700   0.051390 371.297907
soc1                                        -0.035712   0.045810 208.432166
age_m                                        0.159072   0.053104 219.149499
sex1                                         0.046475   0.052473 210.709599
dp_inperson1                                 0.070733   0.075219 210.6054

Computing profile confidence intervals ...



Fitting model for peri_rt...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1182.9

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.3606 -0.5591 -0.0093  0.5635  2.9691 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.05458  0.2336  
 Residual             0.94707  0.9732  
Number of obs: 406, groups:  sub, 227

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                  0.022128   0.074594 211.087626
ICPS_early_MOTOR_diff_collapsed             -0.040103   0.052152 349.150030
soc1                                         0.109990   0.049349 208.791166
age_m                                       -0.007660   0.052185 216.772018
sex1                                         0.005074   0.051147 206.837388
dp_inperson1                                -0.017739   0.073772 208.

Computing profile confidence intervals ...



Fitting model for pes...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1143.3

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.99871 -0.54999  0.06388  0.56085  2.95700 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1848   0.4299  
 Residual             0.7313   0.8552  
Number of obs: 407, groups:  sub, 226

Fixed effects:
                                             Estimate Std. Error         df
(Intercept)                                 -0.063784   0.075314 220.549292
ICPS_early_MOTOR_diff_collapsed              0.038124   0.050666 376.211501
soc1                                        -0.048630   0.043796 210.234044
age_m                                        0.011367   0.052325 222.308316
sex1                                         0.231639   0.051736 214.177088
dp_inperson1                                 0.078548   0.07449

Computing profile confidence intervals ...



## OCC-only

In [41]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'pea',
    'peri_rt',
    'pes'
)

# 2. Define the static right-hand side of your model formula
predictors <- "ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_occ")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

Fitting model for pea...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1127

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.5953 -0.5731  0.0658  0.6159  2.4850 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1173   0.3425  
 Residual             0.8079   0.8989  
Number of obs: 398, groups:  sub, 222

Fixed effects:
                                          Estimate Std. Error        df t value
(Intercept)                               -0.06967    0.07239 200.86644  -0.962
ICPS_early_OCC_diff_collapsed              0.09989    0.04976 368.99960   2.008
soc1                                      -0.04983    0.04591 199.93196  -1.085
age_m                                      0.16030    0.05097 208.02133   3.145
sex1                                       0.05165    0.05098 203.26260   1.013
dp_inperson1                               0.07309 

Computing profile confidence intervals ...



Fitting model for peri_rt...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1128

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.3472 -0.5487 -0.0102  0.5584  2.9888 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.08772  0.2962  
 Residual             0.85031  0.9221  
Number of obs: 396, groups:  sub, 221

Fixed effects:
                                           Estimate Std. Error         df
(Intercept)                                0.046935   0.072364 202.552700
ICPS_early_OCC_diff_collapsed             -0.008986   0.050744 362.143747
soc1                                       0.076631   0.047064 200.166525
age_m                                     -0.022747   0.050881 206.583738
sex1                                      -0.016153   0.050827 201.948008
dp_inperson1                              -0.030267   0.072333 202.249531
ICPS_earl

Computing profile confidence intervals ...



Fitting model for pes...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1106.8

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.03481 -0.56522  0.03304  0.56014  2.59586 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.2193   0.4683  
 Residual             0.6771   0.8228  
Number of obs: 398, groups:  sub, 223

Fixed effects:
                                          Estimate Std. Error        df t value
(Intercept)                               -0.04039    0.07476 207.15070  -0.540
ICPS_early_OCC_diff_collapsed             -0.02886    0.04901 382.13734  -0.589
soc1                                      -0.03497    0.04237 197.32039  -0.825
age_m                                      0.03967    0.05225 211.01710   0.759
sex1                                       0.23780    0.05235 206.87039   4.543
dp_inperson1                           

Computing profile confidence intervals ...



In [42]:
source(sprintf("%s/code/statistics/combine_docs.R", analysis_path))
combine_docs(table_output_path,
            order_pattern = c(
                "power",
                "ITPS",
                "ICPS",
                "ratio_diff_ms_full",
                "ratio_",
                "a_diff_ms_full",
                "a_diff_",
                "pea_ms_full",
                "pea_",
                "peri_rt_ms_full",
                "peri_rt_",
                "pes_ms_full",
                "pes_"
            )
            )

# Behavior

## Load data

In [14]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
# tf_data <- read.csv("/Users/fzaki001/IDENTIFIABLE/tf_data_collapsed_merged.csv")
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/thrive*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}

tf_data <- subset(tf_data, tf_data$acc == 1)
tf_data <- select(tf_data, -acc)

id_cols <- c('sub', 'age_m', 'sex', 'soc', 'dp_inperson')
data_cols <- c(
    'acc_con',
    'acc_incon',
    'rt_con',
    'rt_incon'
)

tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))

tf_data <- tf_data %>%
  pivot_longer(
    cols = c(acc_con, acc_incon, rt_con, rt_incon),
    names_to = c(".value", "congruency"),
    names_sep = "_"
  ) %>%
  mutate(
    congruency = factor(congruency, levels = c("con", "incon")),
    sub = as.factor(sub),
    soc = as.factor(soc)
  )

tf_data$soc <- as.factor(tf_data$soc)
tf_data$sub <- as.factor(tf_data$sub)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$congruency <- as.factor(tf_data$congruency)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", "acc", "rt")] <- as.data.frame(scale(tf_data[c("age_m", "acc", "rt")]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acongruency <- revalue(tf_data$congruency, c("incon" = "Incongruent", "con" = "Congruent"))

contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$congruency) <- -rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))

[1] "The newest file is: /home/data/NDClab/analyses/thrive-theta-ddm//derivatives/csv/s1_r1/thrive_long_s1_r1_20_05_2026_14_17_15.csv"


In [37]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
  "acc",
    "rt"
)

# 2. Define the static right-hand side of your model formula
predictors <- "congruency * soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

Fitting model for acc...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1697.9

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.13652 -0.58347  0.03785  0.59165  2.60742 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.1188   0.3447  
 Residual             0.2884   0.5370  
Number of obs: 896, groups:  sub, 240

Fixed effects:
                         Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)              0.026092   0.041344 229.639131   0.631   0.5286    
congruency1              0.768017   0.017954 647.428050  42.777   <2e-16 ***
soc1                     0.016178   0.018233 684.519007   0.887   0.3752    
age_m                   -0.007254   0.028799 229.378522  -0.252   0.8014    
sex1                     0.050606   0.028780 227.773906   1.758   0.0800 .  
dp_inperson1            -0.041500   0.041400 228.858382  

Computing profile confidence intervals ...



Fitting model for rt...
Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: f
   Data: tf_data

REML criterion at convergence: 1519

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.95576 -0.59131 -0.04996  0.57662  2.63247 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.5761   0.7590  
 Residual             0.1465   0.3827  
Number of obs: 893, groups:  sub, 239

Fixed effects:
                         Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)              0.062730   0.072904 231.179563   0.860  0.39043    
congruency1             -0.429434   0.012822 644.151609 -33.492  < 2e-16 ***
soc1                    -0.018362   0.013206 653.678130  -1.390  0.16487    
age_m                   -0.339679   0.051051 230.504511  -6.654 2.06e-10 ***
sex1                     0.128969   0.050947 230.435619   2.531  0.01203 *  
dp_inperson1            -0.049707   0.073098 230.907046  -0.

Computing profile confidence intervals ...



In [16]:
outcome <- "rt"

# 2. Define the static right-hand side of your model formula
predictors <- "congruency * soc * age_m + sex + dp_inperson + (1 | sub)"
f <- as.formula(paste(outcome, "~", predictors))
model <- lmer(f, data = tf_data)

# View the slopes and significance tests (slope != 0)
summary(simple_slopes, infer = c(TRUE, TRUE), adjust = 'fdr')

# Calculate the difference between the slopes across levels of 'acc'
slope_diffs <- pairs(simple_slopes)

# View the differences and significance tests
summary(slope_diffs, infer = c(TRUE, TRUE))

library(broom)
library(knitr)

# tidy() converts the objects into clean tibbles
tidy_slopes <- tidy(simple_slopes, conf.int = TRUE)
tidy_diffs <- tidy(slope_diffs, conf.int = TRUE)

cat("\n--- Simple Slopes ---\n")
print(kable(tidy_slopes, digits = 3, format = "simple"))

cat("\n--- Slope Differences (e.g., Error - Correct) ---\n")
print(kable(tidy_diffs, digits = 3, format = "simple"))

,congruency,age_m.trend,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,con,-0.3020673,0.05264434,260.4781,-0.4207505,-0.1833841,-5.737887,2.655791e-08
2,incon,-0.3772900,0.05263087,260.2467,-0.4959435,-0.2586366,-7.168607,1.565066e-11


,contrast,estimate,SE,df,lower.CL,upper.CL,t.ratio,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,con - incon,0.07522275,0.02565068,643.9716,0.02485368,0.1255918,2.932583,0.003480939



--- Simple Slopes ---


congruency    age_m.trend   std.error        df   conf.low   conf.high   statistic   p.value
-----------  ------------  ----------  --------  ---------  ----------  ----------  --------
con                -0.302       0.053   260.478     -0.406      -0.198      -5.738         0
incon              -0.377       0.053   260.247     -0.481      -0.274      -7.169         0

--- Slope Differences (e.g., Error - Correct) ---


term         contrast       null.value   estimate   std.error        df   conf.low   conf.high   statistic   p.value
-----------  ------------  -----------  ---------  ----------  --------  ---------  ----------  ----------  --------
congruency   con - incon             0      0.075       0.026   643.972      0.025       0.126       2.933     0.003
